# 📌 Importação de bibliotecas

In [1]:
import numpy as np
import pandas as pd

In [2]:
from typing import Union

In [3]:
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

In [5]:
from scipy.stats import f_oneway
from scipy.stats import levene
from scipy import stats
import pingouin as pg

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate, KFold
from sklearn.model_selection import StratifiedKFold
from imblearn.pipeline import Pipeline as imbpipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import NearMiss
from yellowbrick.model_selection import FeatureImportances

from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_squared_log_error, mean_absolute_percentage_error
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import PrecisionRecallDisplay
from sklearn.metrics import average_precision_score
from sklearn.metrics import classification_report
from yellowbrick.classifier import ClassificationReport
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, auc, precision_recall_curve
)

In [7]:
import copy

In [8]:
import warnings

## Importação de pacotes locais

In [9]:
import scripts.local_tools as lt
import scripts.telecomx_analysis as ta
import scripts.telecomx_machine_learning as tml

## Configurações do ambiente

In [10]:
pd.set_option('display.max_columns', None)

In [11]:
warnings.simplefilter(action='ignore', category=FutureWarning)

## Constantes

In [12]:
NUM_SEMENTE_ALEATORIA = 42
TAMANHO_TESTE = 0.3
MAXIMO_ITERACAO = 1000

In [13]:
LIST_SCORING = ['accuracy','recall', 'precision', 'f1']

# 📌 Extração de dados

In [14]:
df_dados = pd.read_csv('./dados/dados_tratados.csv')

In [15]:
df_dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   customerID                    7043 non-null   object 
 1   Churn                         7043 non-null   int64  
 2   customer_gender               7043 non-null   object 
 3   customer_SeniorCitizen        7043 non-null   int64  
 4   customer_Partner              7043 non-null   int64  
 5   customer_Dependents           7043 non-null   int64  
 6   customer_tenure               7043 non-null   int64  
 7   phone_PhoneService            7043 non-null   int64  
 8   phone_MultipleLines           7043 non-null   int64  
 9   internet_InternetService      7043 non-null   int64  
 10  internet_OnlineSecurity       7043 non-null   int64  
 11  internet_OnlineBackup         7043 non-null   int64  
 12  internet_DeviceProtection     7043 non-null   int64  
 13  int

In [16]:
df_dados.nunique()

customerID                      7043
Churn                              2
customer_gender                    2
customer_SeniorCitizen             2
customer_Partner                   2
customer_Dependents                2
customer_tenure                   72
phone_PhoneService                 2
phone_MultipleLines                2
internet_InternetService           2
internet_OnlineSecurity            2
internet_OnlineBackup              2
internet_DeviceProtection          2
internet_TechSupport               2
internet_StreamingTV               2
internet_StreamingMovies           2
account_Contract                   3
account_PaperlessBilling           2
account_PaymentMethod              4
account_Charges_Monthly         1585
account_Charges_Total           6534
internet_Service_Description       3
customer_tenure_bins               6
account_Charges_Monthly_bins       6
account_Charges_Total_bins        13
account_Contract_Monthly           2
additional_InternetService         7
o

Verificar dados duplicados

In [17]:
df_dados.duplicated().sum()

0

# 📌 Tratamento de dados

In [18]:
df_ohe = tml.df_final_modelo_v1(df_dados)

# 📌 Seleção e validação dos modelos de treinamento

## Tabela de verificação de padronização dos dados

| Modelo                 | Precisa padronizar? | Tipo de padronização (se necessário)   |
| ---------------------- | ------------------- | -------------------------------------- |
| DecisionTreeClassifier | ❌ Não               | —                                      |
| LogisticRegression     | ✅ Sim               | `StandardScaler` (média 0, desvio 1)   |
| RandomForest           | ❌ Não               | —                                      |
| XGBoost                | ⚠️ Opcional         | Pode ajudar (StandardScaler ou MinMax) |
| LightGBM               | ⚠️ Opcional         | Pode ajudar (StandardScaler ou MinMax) |
| CatBoost               | ❌ Não               | —                                      |


## Conceitos sobre as métricas de um modelo

Aqui está o quadro comparativo para churn (evasão de clientes), considerando que a classe positiva é “cliente vai sair”:

| **Métrica**                | **O que mede**                                                                     | **Quando valor é alto**                                        | **Risco quando valor é baixo**                                   | **Custo de erro associado**                                                |
| -------------------------- | ---------------------------------------------------------------------------------- | -------------------------------------------------------------- | ---------------------------------------------------------------- | -------------------------------------------------------------------------- |
| **Precisão (Precision)**   | Entre todos que o modelo previu como “vai sair”, qual porcentagem realmente saiu   | Você gasta retenção apenas em quem de fato sairia              | Gastar recursos em retenção de clientes que iam ficar (FP alto)  | **Custo financeiro** com campanhas desnecessárias (descontos, bônus, etc.) |
| **Recall (Sensibilidade)** | Entre todos que realmente saíram, qual porcentagem o modelo previu como “vai sair” | Você identifica a maior parte dos clientes que iam sair        | Deixar escapar clientes que saem (FN alto)                       | **Perda de receita** e potencial perda de market share                     |
| **F1-Score**               | Média harmônica de precisão e recall                                               | Equilíbrio entre acertar quem vai sair e evitar falsos alarmes | Ou alto custo de retenção inútil ou perda de clientes — ou ambos | **Equilíbrio financeiro e estratégico** — custo total menor                |
| **FP (Falso Positivo)**    | Previu saída, mas o cliente ficaria                                                | —                                                              | Gastar para reter quem não ia sair                               | Desperdício de budget de retenção                                          |
| **FN (Falso Negativo)**    | Previu permanência, mas o cliente saiu                                             | —                                                              | Não agir para reter quem realmente sairia                        | Perda direta de receita + possível impacto na reputação                    |
    

📌 Resumo visual da prioridade

    Se retenção for muito cara → priorizar alta precisão.

    Se perder clientes for muito prejudicial → priorizar alto recall.

    Se quer balancear ambos → otimizar F1-Score.

## Split base de dados em train e test

In [19]:
X = df_ohe.drop(columns=['Churn'])

In [20]:
y = df_ohe.Churn

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = NUM_SEMENTE_ALEATORIA, test_size=TAMANHO_TESTE, stratify=y)

## Treinamento do Modelo - RandomForestClassifier

In [22]:
model_rfc = RandomForestClassifier(max_depth=10, random_state=NUM_SEMENTE_ALEATORIA)
model_rfc.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, random_state=42)

In [23]:
model_rfc.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': 10,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

In [24]:
y_pred_rfc = model_rfc.predict(X_test)

In [25]:
y_pred_proba_rfc = model_rfc.predict_proba(X_test)

### Relatório de métricas

In [26]:
tml.df_classifier_metrics(y_test, y_pred_rfc, y_pred_proba_rfc, ['RandomForestClassifier'])

,accuracy,precision,recall,f1,roc_auc,support,True Negative,False Positive,False Negative,True Positive
RandomForestClassifier,0.784193,0.610063,0.518717,0.560694,0.819857,2113,1366,186,270,291


In [27]:
tml.avaliar_modelo(y_test, y_pred_rfc, y_pred_proba_rfc, print_graph=False)

Métricas:
Acurácia: 0.7842
Precisão: 0.6101
Recall: 0.5187
F1-Score: 0.5607
ROC AUC: 0.8199

Relatório de classificação:

              precision    recall  f1-score   support

           0       0.83      0.88      0.86      1552
           1       0.61      0.52      0.56       561

    accuracy                           0.78      2113
   macro avg       0.72      0.70      0.71      2113
weighted avg       0.78      0.78      0.78      2113


Matrix de confusão:

[[1366  186]
 [ 270  291]]


### Otimizando os hiperparâmetros com o GridSearchCV

#### Teste 1

In [28]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 7],
    'min_samples_leaf': [1, 2],
    'bootstrap': [True, False],
    'class_weight': [None, 'balanced']
}

In [29]:
model_grid_rfc = GridSearchCV(
    estimator=model_rfc,
    param_grid=param_grid,
    scoring='recall',  # ou 'f1', 'roc_auc'
    cv=5,
    n_jobs=-1,
    verbose=2
)

In [30]:
model_grid_rfc.fit(X_train, y_train)

Fitting 5 folds for each of 144 candidates, totalling 720 fits


GridSearchCV(cv=5,
             estimator=RandomForestClassifier(max_depth=10, random_state=42),
             n_jobs=-1,
             param_grid={'bootstrap': [True, False],
                         'class_weight': [None, 'balanced'],
                         'max_depth': [None, 10, 20],
                         'min_samples_leaf': [1, 2],
                         'min_samples_split': [2, 5, 7],
                         'n_estimators': [100, 200]},
             scoring='recall', verbose=2)

In [31]:
model_grid_rfc.best_params_

{'bootstrap': False,
 'class_weight': 'balanced',
 'max_depth': 10,
 'min_samples_leaf': 2,
 'min_samples_split': 7,
 'n_estimators': 100}

In [32]:
y_pred_grid_rfc = model_grid_rfc.predict(X_test)

In [33]:
y_pred_proba_grid_rfc = model_grid_rfc.predict_proba(X_test)

### Relatório de métricas

In [34]:
tml.df_classifier_metrics(y_test, y_pred_grid_rfc, y_pred_proba_grid_rfc, ['RandomForest - Grid'])

,accuracy,precision,recall,f1,roc_auc,support,True Negative,False Positive,False Negative,True Positive
RandomForest - Grid,0.734501,0.5,0.752228,0.600712,0.821741,2113,1130,422,139,422


In [35]:
tml.avaliar_modelo(y_test, y_pred_grid_rfc, y_pred_proba_grid_rfc, False)

Métricas:
Acurácia: 0.7345
Precisão: 0.5000
Recall: 0.7522
F1-Score: 0.6007
ROC AUC: 0.8217

Relatório de classificação:

              precision    recall  f1-score   support

           0       0.89      0.73      0.80      1552
           1       0.50      0.75      0.60       561

    accuracy                           0.73      2113
   macro avg       0.70      0.74      0.70      2113
weighted avg       0.79      0.73      0.75      2113


Matrix de confusão:

[[1130  422]
 [ 139  422]]


In [36]:
list_df = []
colunas_binarias = lt.identify_columns_binary_values(X_test)
for c in colunas_binarias:
    list_df.append(tml.df_specific_confusion_matrix(X_test, y_test, y_pred_grid_rfc, c))
df_independent = pd.concat(list_df, axis=0)

In [37]:
df_independent.reset_index()

,index,filter_value,support,accuracy,negative predictive value,precision,recall,f1-score,TN,FP,FN,TP
0,customer_SeniorCitizen,1,349,0.690544,0.847826,0.587678,0.855172,0.696629,117,87,21,124
1,customer_Partner,1,1032,0.796512,0.893401,0.483607,0.584158,0.529148,704,126,84,118
2,customer_Dependents,1,616,0.827922,0.908382,0.427184,0.483516,0.453608,466,59,47,44
3,phone_MultipleLines,1,883,0.732729,0.875728,0.532609,0.753846,0.624204,451,172,64,196
4,account_PaperlessBilling,1,1213,0.700742,0.875421,0.533118,0.816832,0.645161,520,289,74,330
5,account_Contract_Monthly,1,1182,0.587986,0.791908,0.503589,0.853955,0.633559,274,415,72,421
6,charges_total_bin__inf_96_62_,1,242,0.574380,0.764706,0.543269,0.933884,0.686930,26,95,8,113
7,charges_total_bin__1182_80_3273_68_,1,564,0.739362,0.892265,0.465347,0.706767,0.561194,323,108,39,94
8,charges_total_bin__198_05_347_90_,1,144,0.687500,0.847222,0.527778,0.775510,0.628099,61,34,11,38
9,charges_total_bin__3273_68_4838_38_,1,214,0.780374,0.960000,0.359375,0.793103,0.494624,144,41,6,23


#### Teste 2

In [38]:
model_grid_rfc.best_params_

{'bootstrap': False,
 'class_weight': 'balanced',
 'max_depth': 10,
 'min_samples_leaf': 2,
 'min_samples_split': 7,
 'n_estimators': 100}

In [39]:
dict(sorted(param_grid.items()))

{'bootstrap': [True, False],
 'class_weight': [None, 'balanced'],
 'max_depth': [None, 10, 20],
 'min_samples_leaf': [1, 2],
 'min_samples_split': [2, 5, 7],
 'n_estimators': [100, 200]}

In [40]:
param_grid = {
    'bootstrap': [False],
    'class_weight': ['balanced'],
    'max_depth': [15, 20],
    'min_samples_leaf': [1, 3],
    'min_samples_split': [2, 4, 9],
    'n_estimators': [200, 300],   
}

In [41]:
model_grid_rfc2 = GridSearchCV(
    estimator=model_rfc,
    param_grid=param_grid,
    scoring='recall',  # ou 'f1', 'roc_auc'
    cv=5,
    n_jobs=-1,
    verbose=2
)

In [42]:
model_grid_rfc2.fit(X_train, y_train)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


GridSearchCV(cv=5,
             estimator=RandomForestClassifier(max_depth=10, random_state=42),
             n_jobs=-1,
             param_grid={'bootstrap': [False], 'class_weight': ['balanced'],
                         'max_depth': [15, 20], 'min_samples_leaf': [1, 3],
                         'min_samples_split': [2, 4, 9],
                         'n_estimators': [200, 300]},
             scoring='recall', verbose=2)

In [43]:
model_grid_rfc2.best_params_

{'bootstrap': False,
 'class_weight': 'balanced',
 'max_depth': 15,
 'min_samples_leaf': 3,
 'min_samples_split': 9,
 'n_estimators': 300}

In [44]:
model_grid_rfc.best_params_

{'bootstrap': False,
 'class_weight': 'balanced',
 'max_depth': 10,
 'min_samples_leaf': 2,
 'min_samples_split': 7,
 'n_estimators': 100}

In [45]:
y_pred_grid_rfc2 = model_grid_rfc2.predict(X_test)

In [46]:
y_pred_proba_grid_rfc2 = model_grid_rfc2.predict_proba(X_test)

### Relatório de métricas

In [47]:
tml.df_classifier_metrics(y_test, y_pred_grid_rfc2, y_pred_proba_grid_rfc2, ['RandomForest - Grid'])

,accuracy,precision,recall,f1,roc_auc,support,True Negative,False Positive,False Negative,True Positive
RandomForest - Grid,0.734974,0.500611,0.730838,0.594203,0.814971,2113,1143,409,151,410


### Pipeline para validação Oversampling

### Consolidação das métricas

In [48]:
df_metricas = []

df_metricas.append(tml.df_classifier_metrics(y_test, y_pred_rfc, y_pred_proba_rfc, ['RandomForestClassifier']))

df_metricas.append(tml.df_classifier_metrics(y_test, y_pred_grid_rfc, y_pred_proba_grid_rfc, ['RandomForest - Grid']))

df_metricas.append(tml.df_classifier_metrics(y_test, y_pred_grid_rfc2, y_pred_proba_grid_rfc2, ['RandomForest - Grid 2']))

#df_metricas.append(tml.df_classifier_metrics(y_test, y_pred_over_rfc, y_pred_proba_over_rfc, ['RandomForest - Grid - Oversample']))

df_metricas = pd.concat(df_metricas, axis=0)
df_metricas

,accuracy,precision,recall,f1,roc_auc,support,True Negative,False Positive,False Negative,True Positive
RandomForestClassifier,0.784193,0.610063,0.518717,0.560694,0.819857,2113,1366,186,270,291
RandomForest - Grid,0.734501,0.500000,0.752228,0.600712,0.821741,2113,1130,422,139,422
RandomForest - Grid 2,0.734974,0.500611,0.730838,0.594203,0.814971,2113,1143,409,151,410


In [49]:
df_metricas.to_clipboard()